In [1]:
from physics.simulation import mcfm, msq
from physics.hzz import zz4l
from physics.hstar import c6

import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt

In [2]:
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "xelatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})
lw  = 1.0

In [3]:
filenames = {
    msq.Component.SIG : 'data/zz4l/ggZZ_sig/events.csv',
    msq.Component.BKG : 'data/zz4l/ggZZ_bkg/events.csv',
    msq.Component.INT : 'data/zz4l/ggZZ_int/events.csv',
    msq.Component.SBI : 'data/zz4l/ggZZ_sbi/events.csv',
    'qq' : 'data/zz4l/qqZZ/events.csv',
}

with open('data/zz4l/xsec.json') as f:
    xsec = json.load(f)

xsec = {
    msq.Component.SIG : xsec['gg4l_sig'] * 1.83 * 2,
    msq.Component.BKG : xsec['gg4l_bkg'] * 1.83 * 2,
    msq.Component.INT : xsec['gg4l_int'] * 1.83 * 2,
    msq.Component.SBI : xsec['gg4l_sbi'] * 1.83 * 2,
    'qq' : xsec['qq4l'] * 1.55 * 2,
}

In [ ]:
events_sig = mcfm.from_csv(cross_section=xsec[msq.Component.SIG], file_path = filenames[msq.Component.SIG], n_rows = 1_000_000)
events_sig = zz4l.analyze(events_sig)

events_bkg = mcfm.from_csv(cross_section=xsec[msq.Component.BKG], file_path = filenames[msq.Component.BKG], n_rows = 1_000_000)
events_bkg = zz4l.analyze(events_bkg)

events_int = mcfm.from_csv(cross_section=xsec[msq.Component.INT], file_path = filenames[msq.Component.INT], n_rows = 1_000_000)
events_int = zz4l.analyze(events_int)

events_sbi = mcfm.from_csv(cross_section=xsec[msq.Component.SBI], file_path=filenames[msq.Component.SBI], n_rows = 1_000_000)
events_sbi = zz4l.analyze(events_sbi)

events_qq = mcfm.from_csv(cross_section=xsec['qq'], file_path=filenames['qq'], n_rows = 1_000_000)
events_qq = zz4l.analyze(events_qq)

In [ ]:
nxbins = 41
xmin, xmax = 180, 1000.0
xbins = np.linspace(xmin, xmax, nxbins + 1)
xcenters = 0.5 * (xbins[:-1] + xbins[1:])
xwidths = np.diff(xbins)

In [6]:
xobs_sig = events_sig.kinematics['4l_mass']
xobs_bkg = events_bkg.kinematics['4l_mass']
xobs_int = events_int.kinematics['4l_mass']
xobs_sbi = events_sbi.kinematics['4l_mass']
xobs_qq  = events_qq.kinematics['4l_mass']

h_sig = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sig.fill(xobs_sig, weight=events_sig.weights)

h_bkg = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_bkg.fill(xobs_bkg, weight=events_bkg.weights)

h_int = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_int.fill(xobs_int, weight=events_int.weights)

h_sbi = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_sbi.fill(xobs_sbi, weight=events_sbi.weights)

h_qq = hist.Hist(hist.axis.Variable(xbins), storage=hist.storage.Weight())
h_qq.fill(xobs_qq, weight=events_qq.weights)

Hist(Variable(array([ 180.,  200.,  220.,  240.,  260.,  280.,  300.,  320.,  340.,
        360.,  380.,  400.,  420.,  440.,  460.,  480.,  500.,  520.,
        540.,  560.,  580.,  600.,  620.,  640.,  660.,  680.,  700.,
        720.,  740.,  760.,  780.,  800.,  820.,  840.,  860.,  880.,
        900.,  920.,  940.,  960.,  980., 1000.]), label='Axis 0'), storage=Weight()) # Sum: WeightedSum(value=43.6998, variance=0.000982657) (WeightedSum(value=43.7586, variance=0.00099032) with flow)

In [23]:
fig, (ax1, ax2) = plt.subplots(2,1, gridspec_kw={'height_ratios': [3,1]}, figsize=(6,5), sharex=True)

l_sbi = ax1.stairs(h_sbi.values(), xbins, color='black', linestyle='-',  linewidth=lw)
l_sig = ax1.stairs(h_sig.values(), xbins, color='red',   linestyle='--', linewidth=lw)
l_bkg = ax1.stairs(h_bkg.values(), xbins, color='grey',  linestyle='--', linewidth=lw)
l_qq  = ax1.stairs(h_qq.values(),  xbins, color='orange', linestyle='-', linewidth=lw)

l_sig.set_label('$gg,\\ |\\mathcal{M}_{\\mathrm{s}}|^2$')
l_bkg.set_label('$gg,\\ |\\mathcal{M}_{\\mathrm{b}}|^2$')
l_sbi.set_label('$gg,\\ |\\mathcal{M}_{\\mathrm{s}} + \\mathcal{M}_{\\mathrm{b}}|^2$')
l_qq.set_label('$q\\bar{q}$')
ax1.legend(frameon=False, fontsize=12)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{d}\sigma / \\mathrm{d} m_{ZZ}\\ \\mathrm{[fb / GeV]}$', loc='top', fontsize=15)
ax1.set_ylim(2e-4, 20.0)
ax1.yaxis.set_ticks([ 2e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0])
ax1.yaxis.set_ticklabels([ '', '$10^{-3}$', '$10^{-2}$', '$0.1$', '$1.0$', '$10$'])
ax1.text(0.00 ,0.03, '$\\approx$', transform=ax1.transAxes, ha='center', va='center', fontsize=12, bbox={'facecolor':'white', 'edgecolor':'none', 'pad':0.0})

ax1.tick_params(axis='x', direction='in')

ax1.text(0.04 ,0.12, '$pp (\\rightarrow h^{\\ast}) \\rightarrow ZZ$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)
ax1.text(0.04 ,0.04, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax1.transAxes, ha='left', va='bottom', fontsize=12)

l_int = ax2.stairs(h_int.values(), xbins, color='royalblue',  linestyle='--', linewidth=lw)
l_int.set_label('$gg,\\ 2\\mathrm{Re}(\\mathcal{M}^{\\dag}_{\\mathrm{s}} \\mathcal{M}_{\\mathrm{b}})$')
ax2.legend(frameon=False, fontsize=12, loc='lower right')

ax2.set_yscale('linear')
# ax2.tick_params(top=True, labeltop=False)

ax2.set_xlim(xmin, xmax)
ax2.xaxis.set_ticks([ 180, 250, 500, 750, 1000 ])
ax2.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)

ax2.set_ylim(-0.08, 0.0)
ax2.yaxis.set_ticks([-0.08, -0.04, 0.0])
ax2.yaxis.set_ticklabels(['$-0.08$', '$-0.04$', '$0$'])

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

fig.tight_layout()
fig.subplots_adjust(hspace=0.00)

fig.canvas.draw()  # update positions

plt.savefig('pp4l_mzz_sm.pdf')
plt.close()